In [17]:
import xarray as xr
import pandas as pd
import dask
import glob
import os
import math
from tqdm.notebook import tqdm


In [18]:
%run ./common_vars.py

In [19]:
var_path = "/glade/work/qingyuany/camml_v6/emulated_vars/*mean*"
mean_paths = glob.glob(var_path, recursive=True)
mean_paths

['/glade/work/qingyuany/camml_v6/emulated_vars/gp_mean_FSNTOA_zonal_lat_2.csv',
 '/glade/work/qingyuany/camml_v6/emulated_vars/gp_mean_FSNTOA_zonal_lat_57.csv',
 '/glade/work/qingyuany/camml_v6/emulated_vars/gp_mean_PRECT_zonal_lat_-47.csv',
 '/glade/work/qingyuany/camml_v6/emulated_vars/gp_mean_FLUT_zonal_lat_-62.csv',
 '/glade/work/qingyuany/camml_v6/emulated_vars/gp_mean_LWCF_zonal_lat_-27.csv',
 '/glade/work/qingyuany/camml_v6/emulated_vars/gp_mean_PRECT_zonal_lat_52.csv',
 '/glade/work/qingyuany/camml_v6/emulated_vars/gp_mean_TMQ_10_15_90_95.csv',
 '/glade/work/qingyuany/camml_v6/emulated_vars/gp_mean_LWCF_zonal_lat_47.csv',
 '/glade/work/qingyuany/camml_v6/emulated_vars/gp_mean_LWCF_zonal_lat_42.csv',
 '/glade/work/qingyuany/camml_v6/emulated_vars/gp_mean_FLUT_zonal_lat_77.csv',
 '/glade/work/qingyuany/camml_v6/emulated_vars/gp_mean_LWCF_5_10_200_220.csv',
 '/glade/work/qingyuany/camml_v6/emulated_vars/gp_mean_TGCLDLWP_zonal_lat_37.csv',
 '/glade/work/qingyuany/camml_v6/emulated_

In [20]:
ppe_zonal = pd.read_csv("/glade/work/qingyuany/camml_v6/zonal_manual_climatologies/ppe_zonal_manualselect.csv", index_col = 0)
obs_zonal = pd.read_csv("/glade/work/qingyuany/camml_v6/zonal_manual_climatologies/obs_zonal_manualselect.csv", index_col = 0)

In [21]:
threshold_level = 1.8

In [15]:
tf_masks = []

for path in tqdm(mean_paths):
    var_name_file = path.split("/")[-1].split("mean_")[1]

    if var_name_file.split("_")[0] == "CLDTOT":
        cli_temp = var_name_file.split("_")[0] + "_" + var_name_file.split("_")[1]

    else:
        cli_temp = var_name_file.split("_")[0]
    
    
    var_name = var_name_file.split(".")[0]
    
    emulated_mean = pd.read_csv(path,index_col=0)
    emulated_std = pd.read_csv("/glade/work/qingyuany/camml_v6/emulated_vars/" + "gp_std_" + var_name_file, index_col=0)
    
    
    obs_temp = obs_zonal.loc[var_name].values[0]
    y_ppe = ppe_zonal[var_name]

    yscale = y_ppe.std()
    ymu = y_ppe.mean()            
    emulated_mean = emulated_mean * yscale + ymu
    emulated_std = emulated_std * yscale

    print(var_name)
    
    if var_name.split("_")[0] == "PRECT":
        emulated_mean = emulated_mean * 1000 * 3600 * 24
        emulated_std = emulated_std * 1000 * 3600 * 24

    
    temp_tf_mask = ((emulated_mean - threshold_level * emulated_std) < obs_temp) & ((emulated_mean + threshold_level * emulated_std) > obs_temp)
    tf_masks.append(temp_tf_mask)



  0%|          | 0/274 [00:00<?, ?it/s]

FSNTOA_zonal_lat_2
FSNTOA_zonal_lat_57
PRECT_zonal_lat_-47
FLUT_zonal_lat_-62
LWCF_zonal_lat_-27
PRECT_zonal_lat_52
TMQ_10_15_90_95
LWCF_zonal_lat_47
LWCF_zonal_lat_42
FLUT_zonal_lat_77
LWCF_5_10_200_220
TGCLDLWP_zonal_lat_37
SWCF_zonal_lat_77
LWCF_zonal_lat_-17
LWCF_zonal_lat_-82
TMQ_zonal_lat_72
TGCLDLWP_zonal_lat_-57
PRECT_0_5_90_95
LWCF_zonal_lat_7
LWCF_zonal_lat_-32
FLUT_0_5_85_90
TGCLDLWP_zonal_lat_12
LWCF_6_8_275_280
TMQ_zonal_lat_52
TMQ_zonal_lat_42
SWCF_7_9_235_240
TMQ_zonal_lat_22
LWCF_zonal_lat_2
TGCLDLWP_zonal_lat_2
SWCF_zonal_lat_82
TMQ_zonal_lat_-77
SWCF_zonal_lat_-62
PRECT_zonal_lat_77
PRECT_zonal_lat_57
FSNTOA_zonal_lat_82
TGCLDLWP_zonal_lat_17
PRECT_zonal_lat_7
LWCF_zonal_lat_72
TGCLDLWP_zonal_lat_-2
CLDTOT_ISCCP_zonal_lat_57
SWCF_zonal_lat_-82
FLUT_zonal_lat_82
LWCF_zonal_lat_-77
TMQ_zonal_lat_-12
TGCLDLWP_zonal_lat_77
CLDTOT_ISCCP_zonal_lat_17
TGCLDLWP_zonal_lat_52
TMQ_zonal_lat_-37
FLUT_zonal_lat_-67
LWCF_zonal_lat_-52
LWCF_zonal_lat_22
CLDTOT_ISCCP_zonal_lat_-7
LWC

In [16]:
pd.concat(tf_masks, axis = 1).to_csv("/glade/work/qingyuany/camml_v6/tf_masks/zonal_masks_" + str(threshold_level) + ".csv")

FSNTOA_zonal_lat_0            873543
PRECT_zonal_lat_20            704842
LWCF_zonal_lat_60            1000000
FSNTOA_zonal_lat_50           976607
TMQ_10_15_90_95              1000000
                              ...   
TGCLDLWP_zonal_lat_-40             0
TMQ_zonal_lat_0               875902
CLDTOT_ISCCP_zonal_lat_60       3986
SWCF_zonal_lat_80            1000000
TGCLDLWP_zonal_lat_80              0
Length: 148, dtype: int64